# Qwen3-4B → Qwen3-1.7B
Без thinking, 1024 токена на агента. Kernel: **AsyncLLM (uv)**.


In [ ]:
import gc
import os
import re
import sys
import time
from contextlib import aclosing
from pathlib import Path
from tempfile import TemporaryDirectory

from IPython.display import Markdown, display

ROOT = Path.cwd().parents[1] if Path.cwd().name == "parallel_thinking" else Path.cwd()
os.chdir(ROOT)
for path in (ROOT / "python", ROOT / "scripts/async_thoughts", ROOT / "scripts/parallel_thinking"):
    sys.path.insert(0, str(path))
os.environ.update(
    CUDA_DEVICE_ORDER="PCI_BUS_ID",
    CUDA_VISIBLE_DEVICES="2",
    TVM_FFI_CUDA_ARCH_LIST="8.6",
    OMP_NUM_THREADS="8",
)
os.environ["PATH"] = f"{Path(sys.executable).parent}:/usr/local/cuda/bin:" + os.environ["PATH"]

import torch
from async_helpers import isolated, one_token, recorded_example, together
from async_thoughts.training import DemoRecorder
from hogwild_formatting import MathFormatting
from minisgl.core import SamplingParams
from minisgl.llm import AsyncLLM
from minisgl.shared_cache import AsyncContext
from minisgl.training.replay import Replay
from minisgl.training.sft import evaluate, supervised_loss
from transformers import AutoModelForCausalLM, AutoTokenizer

TEACHER = "models/Qwen3-4B"
STUDENT = "models/Qwen3-1.7B"
WORKERS = ("Alice", "Bob")
PROBLEM = (
    "Three vertices of a cube are $P=(7,12,10)$ , $Q=(8,8,1)$ , and "
    "$R=(11,3,9)$ . What is the surface area of the cube?"
)

## Генерация учителя


In [ ]:
async def solve_parallel(model, *, max_steps=1024, prompt_ids=None):
    started = time.perf_counter()
    llm = AsyncLLM(
        str(model),
        dtype=torch.bfloat16,
        memory_ratio=0.65,
        max_running_req=4,
        max_seq_len_override=16384,
        cuda_graph_bs=[1, 2],
        cuda_graph_max_bs=2,
        shared_cuda_graph_max_depth=6,
        shared_cuda_graph_prefill_rows=[32, 256],
        max_prefill_rows=256,
        distributed_addr="tcp://127.0.0.1:0",
    )
    llm = DemoRecorder(llm)
    blocks = []
    try:
        tokenizer = llm.tokenizer
        encode = lambda text: tokenizer.encode(text, add_special_tokens=False)

        fmt = MathFormatting(
            tokenizer, WORKERS, enable_thinking=False
        )
        for _ in range(6):
            blocks.append(await llm.create_block())
        common, others_header, own_header, alice, bob, answer_block = blocks
        worker_blocks = [alice, bob]
        contexts = [
            AsyncContext(cache_view=[common, others_header, bob, own_header, alice]),
            AsyncContext(cache_view=[common, others_header, alice, own_header, bob]),
        ]
        initial_ids = (
            list(prompt_ids) if prompt_ids is not None else encode(fmt.apply_chat_template(PROBLEM))
        )

        await llm.forward(initial_ids, write_to=common, return_logits=False)
        await together(
            llm.forward(
                encode(fmt.current_step_header), write_to=others_header, return_logits=False
            ),
            llm.forward(
                encode(fmt.current_worker_header), write_to=own_header, return_logits=False
            ),
        )

        prefixes = [
            f"{fmt.get_step_prefix(WORKERS[0], 1)}Hi, I'm {WORKERS[0]}. Here's how we can collaborate",
            f"{fmt.get_step_prefix(WORKERS[1], 1)}Hi, I'm {WORKERS[1]}.",
        ]
        current = [encode(prefix) for prefix in prefixes]

        await together(
            *(
                llm.forward(
                    ids[:-1],
                    cache_view=[common, others_header, own_header, block],
                    return_logits=False,
                )
                for ids, block in zip(current, worker_blocks)
            )
        )
        for ctx, ids in zip(contexts, current):
            ctx.next_input_id = ids[-1]

        history, events = [], []
        step_indices, generated = [1, 1], [0, 0]
        sampled = [[], []]
        forbidden = [
            tokenizer.vocab[t]
            for t in ("<|im_start|>", "<|endoftext|>")
            if t in tokenizer.vocab
        ]

        eos = tokenizer.eos_token_id
        forbidden = [token for token in forbidden if token != eos]
        generation_started = time.perf_counter()

        for iteration in range(1, max_steps + 1):
            new_tokens = await together(*(one_token(llm, ctx, forbidden) for ctx in contexts))
            for i, token in enumerate(new_tokens):
                current[i].append(token)
                generated[i] += 1
                sampled[i].append(token)

            finished = [i for i in range(2) if fmt.is_end_of_step(current[i])]
            await together(
                *(llm.forward(cache_view=contexts[i], return_logits=False) for i in finished)
            )
            for i in finished:
                block = worker_blocks[i]
                assert block.token_ids == current[i], "A boundary token was lost or duplicated."
                await llm.append_block(common, block)
                history.extend(current[i])
                assert common.token_ids == initial_ids + history
                events.append(
                    {
                        "round": iteration,
                        "worker": WORKERS[i],
                        "step": step_indices[i],
                        "tokens": len(current[i]),
                        "text": tokenizer.decode(current[i]),
                    }
                )
                await llm.free_block(block)
                assert block.num_tokens == 0
                step_indices[i] += 1
                start = fmt.get_step_prefix(WORKERS[i], step_indices[i])
                current[i] = encode(start)

            await together(
                *(
                    llm.forward(current[i][:-1], cache_view=contexts[i], return_logits=False)
                    for i in finished
                )
            )
            for i in finished:
                contexts[i].next_input_id = current[i][-1]

            for i, ctx in enumerate(contexts):
                assert ctx.output_block.token_ids == current[i][:-1]
                assert ctx.next_input_id == current[i][-1]
            if iteration % 128 == 0:
                print(f"round {iteration}: {len(events)} completed steps", flush=True)
            if eos in new_tokens:
                break

        await together(*(llm.forward(cache_view=ctx, return_logits=False) for ctx in contexts))
        for i, ctx in enumerate(contexts):
            assert ctx.output_block.token_ids == current[i]
        parallel_seconds = time.perf_counter() - generation_started
        transcript = fmt.history_header + fmt.sep + tokenizer.decode(history)
        transcript += fmt.sep + fmt.current_step_header
        transcript += "".join(
            tokenizer.decode(ids) + fmt.incomplete_step + fmt.sep for ids in current
        )

        suffix = fmt.s1_finisher_suffix
        answer_ids = encode(suffix)
        answer_ctx = AsyncContext(
            cache_view=[common, others_header, alice, own_header, bob, answer_block]
        )
        await llm.forward(answer_ids[:-1], cache_view=answer_ctx, return_logits=False)
        answer_ctx.next_input_id = answer_ids[-1]
        answer_tokens = []
        async with aclosing(
            llm.async_generate(
                answer_ctx,
                max_steps=64,
                forbid_ids=forbidden,
                sampling_params=SamplingParams(temperature=0.0, max_tokens=64),
            )
        ) as stream:
            async for token in stream:
                if token == eos:
                    break
                answer_tokens.append(token)
                if fmt.get_final_answer(suffix + tokenizer.decode(answer_tokens)) is not None:
                    break
        answer_text = suffix + tokenizer.decode(answer_tokens)
        result = {
            "model": str(model),
            "problem": PROBLEM,
            "rounds": iteration,
            "generated_tokens": dict(zip(WORKERS, generated)),
            "completed_steps": len(events),
            "merge_events": events,
            "transcript": transcript,
            "answer_text": answer_text,
            "answer": fmt.get_final_answer(answer_text),
            "parallel_seconds": parallel_seconds,
            "total_seconds": time.perf_counter() - started,
        }
        result["generated_text"] = {w: tokenizer.decode(ids) for w, ids in zip(WORKERS, sampled)}
        result["prompt_ids"] = initial_ids
        result["history_ids"] = history
        result["current_ids"] = current
        result["answer_ids"] = answer_tokens
        result["example"] = recorded_example(llm)
        return result
    finally:

        try:
            for block in blocks:
                await llm.free_block(block)
        finally:
            await llm.close()
            gc.collect()
            torch.cuda.empty_cache()

In [ ]:
teacher = isolated(solve_parallel, TEACHER)
example = teacher.pop("example")
display(Markdown(teacher["transcript"]))
print("Teacher answer:", teacher["answer"], "Published blocks:", teacher["completed_steps"])
role_headers = {
    worker: re.findall(r"\*\*(?:Alice|Bob) \[\d+\]:\*\*", text)
    for worker, text in teacher["generated_text"].items()
}
print("Model-generated role headers:", role_headers)
assert teacher["answer"] == 294
assert not any(role_headers.values()), "Inspect the teacher: a worker is writing dialogue headers."



## Тренировочный пример


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT)
trace, targets = example["trace"], torch.tensor(example["targets"])
print("PROMPT\n" + tokenizer.decode(teacher["prompt_ids"]))
print("\nTRAINING TARGETS (grouped by output block; prefill/flush are masked)")
for block in sorted({token.block for token in trace.tokens}):
    ids = [
        target
        for token, target in zip(trace.tokens, example["targets"])
        if token.block == block and target != -100
    ]
    if ids:
        print(f"\nBlock {block}, {len(ids)} target tokens:\n{tokenizer.decode(ids)}")
print("\nFirst 8 supervised rows: input → target; position; visible (block, length)")
rows = [
    (token, target) for token, target in zip(trace.tokens, example["targets"]) if target != -100
]
for token, target in rows[:8]:
    print(
        repr(tokenizer.decode([token.token_id])),
        "→",
        repr(tokenizer.decode([target])),
        "position=",
        token.position,
        "view=",
        token.view,
    )
print("Total supervised tokens:", len(rows))

## Обучение
Master-веса и AdamW — BF16 на CPU, вычисления — BF16 на GPU.
Replay восстанавливает видимость и позиции; loss — средняя CE по целевым токенам.


In [ ]:
torch.manual_seed(7)
replay = Replay.from_trace(
    trace, query_chunk_size=64, attention_checkpointing=True, cache_plans=True
)
student = AutoModelForCausalLM.from_pretrained(STUDENT, dtype=torch.bfloat16).cuda()
student.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

master = {
    name: torch.nn.Parameter(p.detach().cpu().clone()) for name, p in student.named_parameters()
}
optimizer = torch.optim.AdamW(master.values(), lr=3e-5, foreach=False)

def evaluate_student():
    metrics = evaluate(student.eval(), [(replay, targets)], chunk_size=64)
    return {key: metrics[key] for key in ("loss", "accuracy")}

print(
    "Master:",
    next(iter(master.values())).dtype,
    "RoPE frequencies:",
    student.model.rotary_emb.inv_freq.dtype,
)
initial = evaluate_student()
print("Initial:", initial)

In [ ]:
for step in range(1, 81):
    student.train()
    student.zero_grad(set_to_none=True)
    optimizer.zero_grad(set_to_none=True)
    loss = supervised_loss(student, replay, targets, chunk_size=64) / targets.ne(-100).sum()
    loss.backward()
    for name, parameter in student.named_parameters():
        master[name].grad = parameter.grad.detach().cpu()
        parameter.grad = None
    torch.nn.utils.clip_grad_norm_(master.values(), 1.0, error_if_nonfinite=True)
    optimizer.step()
    with torch.no_grad():
        for name, parameter in student.named_parameters():
            parameter.copy_(master[name])
    print(f"Step {step}: loss={loss.item():.6f}")
    if step % 5 == 0:
        final = evaluate_student()
        print("Evaluation:", final)
        if final["accuracy"] == 1.0 and final["loss"] < 0.01:
            break
print("Initial:", initial, "Final:", final)
print("Adam moments:", next(iter(optimizer.state.values()))["exp_avg"].dtype)


## Свободная генерация
Токены и публикации учителя не подставляются.


In [ ]:
student.cpu()
gc.collect()
torch.cuda.empty_cache()
with TemporaryDirectory() as checkpoint:
    student.save_pretrained(checkpoint)
    tokenizer.save_pretrained(checkpoint)
    result = isolated(solve_parallel, checkpoint, prompt_ids=teacher["prompt_ids"])
display(Markdown(result["transcript"]))
print("Student answer:", result["answer"])
print(
    "Same tokens:",
    all(result[key] == teacher[key] for key in ("history_ids", "current_ids", "answer_ids")),
)
print("Same publications:", result["merge_events"] == teacher["merge_events"])
display(Markdown(result["answer_text"]))